# 10 梯度提升 Gradient Boosting

Gradient Boosting 也是 boosting，但它的直觉是：每一轮新模型都去拟合当前模型还没解释掉的残差或负梯度。


## 0. 学习目标和阅读地图

Gradient Boosting 的核心是“不断修正当前模型的错误”。你需要掌握：

1. 为什么平方误差下新树拟合的是残差。
2. 为什么学习率和树数量要一起看。
3. 它和 AdaBoost 的共同点和差异。
4. 为什么现代 GBDT 是表格数据强基线。


## 1. 数学逻辑

模型是多个弱模型相加：

$$F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta h_m(x)$$

平方误差回归里，负梯度就是残差：

$$r_i = y_i - F_{m-1}(x_i)$$

所以每一轮训练一棵小树拟合残差，然后加回原模型。


## 1.1 推导拆开看：残差就是负梯度

平方误差：

$$L=\frac{1}{2}(y-F(x))^2$$

对当前预测 `F(x)` 求导：

$$\frac{\partial L}{\partial F}=F(x)-y$$

负梯度是：

$$y-F(x)$$

这就是残差。所以在平方误差回归里，Gradient Boosting 每轮训练一棵树去拟合当前残差。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)
X = np.linspace(-3, 3, 180).reshape(-1, 1)
y = np.sin(X[:, 0]) + 0.3 * X[:, 0] + np.random.normal(scale=0.18, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


## 1.2 模型如何逐步变复杂

第一轮通常只预测均值。之后每一棵小树只负责解释剩余误差的一部分：

$$F_m(x)=F_{m-1}(x)+\eta h_m(x)$$

`eta` 越小，每棵树贡献越保守，通常需要更多树。


In [ ]:
# 从零实现平方误差下的 gradient boosting：不断拟合残差
learning_rate = 0.2
pred_train = np.full(len(y_train), y_train.mean())
pred_test = np.full(len(y_test), y_train.mean())
trees = []

for m in range(20):
    residual = y_train - pred_train
    tree = DecisionTreeRegressor(max_depth=2, random_state=m)
    tree.fit(X_train, residual)
    pred_train += learning_rate * tree.predict(X_train)
    pred_test += learning_rate * tree.predict(X_test)
    trees.append(tree)
    if m in [0, 1, 4, 9, 19]:
        print(f'round {m+1:2d} | test MSE={mean_squared_error(y_test, pred_test):.4f}')


## 1.3 从零实现代码怎么读

从零版本的关键变量：

- `pred_train`：当前集成模型对训练集的预测。
- `residual = y_train - pred_train`：当前还没解释掉的部分。
- `tree.fit(X_train, residual)`：新树学习残差。
- `pred_train += learning_rate * tree.predict(...)`：把新树加入集成。


In [ ]:
model = GradientBoostingRegressor(n_estimators=80, learning_rate=0.08, max_depth=2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('GradientBoostingRegressor MSE:', round(mean_squared_error(y_test, pred), 4))

line_x = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.scatter(X_train[:,0], y_train, s=20, alpha=0.6, label='train')
plt.plot(line_x[:,0], model.predict(line_x), color='red', label='boosting fit')
plt.legend()
plt.title('Gradient Boosting 拟合非线性函数')
plt.show()


In [ ]:
# 诊断：用 staged_predict 观察每轮新增树后的 MSE
staged_mse = [mean_squared_error(y_test, pred) for pred in model.staged_predict(X_test)]
plt.plot(range(1, len(staged_mse) + 1), staged_mse)
plt.title('Gradient Boosting 轮数与测试 MSE')
plt.xlabel('n_estimators')
plt.ylabel('test MSE')
plt.show()
print('最佳 MSE:', round(min(staged_mse), 4), '出现在第', int(np.argmin(staged_mse) + 1), '轮')


## 2.1 如何诊断 Gradient Boosting

如果训练误差持续下降但测试误差开始上升，就是过拟合。常用控制方法：

- 降低 `learning_rate`。
- 限制 `max_depth`。
- 使用早停。
- 限制叶子节点样本数。


## 2. 常见误区

- Gradient Boosting 很强，但超参数敏感，容易过拟合。
- `learning_rate` 小通常要配更多树。
- 树太深时，每轮弱学习器不再“弱”，可能过拟合。

## 3. 小实验

- 改 `learning_rate` 和 `n_estimators` 的组合。
- 改 `max_depth`，观察曲线是否变得锯齿化。
- 把噪声调大，观察过拟合。


## 5. 复习清单

- Gradient Boosting 是加法模型。
- 平方误差下，每轮拟合残差。
- 学习率小更稳，但需要更多树。
- 它强大但更需要调参和验证集。
